# GNN H1 Ensemble Training (Spatiotemporal Split, Local AEM Graph 2.5 km)

This notebook uses the current H1 mainline on the 2011-2023 shallow-only WTD inputs. It rebuilds the H1 spatiotemporal sample tables and parent graph, trains the 5-seed ensemble, and summarizes performance with `Pearson r`, `RMSE`, and `NSE`.


In [ ]:
from pathlib import Path
import shutil
import sys
import time

import pandas as pd
import torch

NOTEBOOK_DIR = Path.cwd()
REPO_ROOT = NOTEBOOK_DIR if (NOTEBOOK_DIR / 'src').exists() else NOTEBOOK_DIR.parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.utils import load_config, processed_paths, save_month_index
from src.preprocess_swb import preprocess_swb
from src.preprocess_aiwum import preprocess_aiwum
from src.preprocess_wtd import preprocess_wtd
from src.build_samples import build_sample_table
from src.build_graphs import build_aem_graph
from src.train import train_experiment
from src.evaluate import evaluate_experiment
from src.conformal import fit_normalized_conformal, apply_normalized_conformal, summarize_prediction_intervals


In [ ]:
BASE_CONFIG_PATH = REPO_ROOT / 'configs/GNN_H1.yaml'
BASE_GRAPH_DIR = REPO_ROOT / 'outputs/GNN_H1/graphs'
ENSEMBLE_PARENT = REPO_ROOT / 'outputs/GNN_H1'
EXPERIMENT_NAME = 'aem_gnn'
SEEDS = [11, 22, 33, 44, 55]
INTERVAL_LEVELS = (0.75,)

ENSEMBLE_PARENT


In [ ]:
print('Notebook Python:', sys.executable)
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA version:', torch.version.cuda)
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('This kernel is using CPU-only torch. Select the project .venv kernel after installing CUDA torch, or reinstall torch with the CUDA wheel.')


## Rebuild Spatiotemporal Inputs and Graphs

Run this once before training. It refreshes the H1 sample tables for the current spatiotemporal grid-time split and rebuilds the parent AEM graph used by the ensemble members.


In [ ]:
def make_abs_config(config):
    config = dict(config)
    config['paths'] = dict(config['paths'])
    for key, value in list(config['paths'].items()):
        pth = Path(value)
        if not pth.is_absolute():
            config['paths'][key] = str((REPO_ROOT / pth).resolve())
    return config


def safe_unlink(path):
    path = Path(path)
    if path.exists():
        path.unlink()


def safe_rmtree(path):
    path = Path(path)
    if not path.exists():
        return
    resolved = path.resolve()
    workspace = REPO_ROOT.resolve()
    if workspace not in resolved.parents and resolved != workspace:
        raise RuntimeError(f'Refusing to remove path outside workspace: {resolved}')
    shutil.rmtree(resolved)


def rebuild_inputs_and_parent_graphs():
    config = make_abs_config(load_config(BASE_CONFIG_PATH))
    if str(config['data'].get('split_by', '')).lower() != 'spatiotemporal_grid_time':
        raise ValueError(f'{BASE_CONFIG_PATH.name} must use split_by: spatiotemporal_grid_time for this notebook.')

    paths = processed_paths(config)

    # Remove only H-specific generated training inputs/caches so this notebook cannot reuse stale artifacts.
    for key in [
        'swb_monthly_csv',
        'swb_metadata_json',
        'aiwum_monthly_csv',
        'aiwum_metadata_json',
        'wtd_monthly_csv',
        'wtd_metadata_json',
        'dynamic_cache_pt',
        'sample_table_csv',
        'sample_metadata_json',
        'month_index_csv',
        'time_index_csv',
        'scalers_pt',
    ]:
        safe_unlink(paths[key])
    for split_table in paths['root'].glob('sample_table_*.csv'):
        safe_unlink(split_table)
    safe_rmtree(paths['neighborhood_cache_dir'])
    safe_rmtree(ENSEMBLE_PARENT / 'graphs')

    save_month_index(config)
    preprocess_swb(config)
    preprocess_aiwum(config)
    preprocess_wtd(config)
    sample_result = build_sample_table(config)
    graph_outputs = {'aem': build_aem_graph(config)}

    split_files = {
        split: str(paths['root'] / f'sample_table_{split}.csv')
        for split in ['train', 'validation', 'test_temporal', 'test_spatial', 'test']
        if (paths['root'] / f'sample_table_{split}.csv').exists()
    }

    return {
        'processed_root': str(paths['root']),
        'sample_count': sample_result.metadata['sample_count'],
        'split_counts': sample_result.metadata['split_counts'],
        'split_windows': {
            'train_end': config['data']['train_end'],
            'validation': [config['data']['val_start'], config['data']['val_end']],
            'test': [config['data']['test_start'], config['data']['test_end']],
        },
        'split_files': split_files,
        'graphs': {'aem': str(graph_outputs['aem'])} if 'aem' in graph_outputs else {name: str(path) for name, path in graph_outputs.items()},
    }


setup_summary = rebuild_inputs_and_parent_graphs()
setup_summary


## Train Spatiotemporal Ensemble Members

This cell trains one model per seed on the spatiotemporal training split and evaluates on validation, temporal test, spatial test, and combined test samples. It seeds each output directory with the prebuilt AEM graph so we do not rebuild it every time.


In [ ]:
def metric_from_result(result, split, metric):
    overall = result['overall'] if isinstance(result, dict) else result.overall
    rows = overall[overall['split'].astype(str) == split]
    if rows.empty or metric not in rows.columns:
        return None
    return float(rows.iloc[0][metric])


def configure_runtime_flags(config):
    cuda_available = torch.cuda.is_available()
    config['training']['pin_memory'] = bool(config['training'].get('pin_memory', True)) and cuda_available
    config['training']['use_amp'] = bool(config['training'].get('use_amp', True)) and cuda_available
    return config


SPLITS_TO_SUMMARIZE = ['train', 'validation', 'test_temporal', 'test_spatial', 'test']
METRICS_TO_SUMMARIZE = ['pearson_r', 'rmse', 'nse']

run_rows = []

for seed in SEEDS:
    config = make_abs_config(load_config(BASE_CONFIG_PATH))
    config = configure_runtime_flags(config)
    config['training']['seed'] = int(seed)
    seed_root = ENSEMBLE_PARENT / f'seed{seed}'
    config['paths']['output_dir'] = str(seed_root.resolve())

    print(f"Training seed {seed} | cuda={torch.cuda.is_available()} | pin_memory={config['training']['pin_memory']} | amp={config['training']['use_amp']}")

    # Always retrain all five seeds for the current spatiotemporal sample table.
    for rel in ['checkpoints', 'metrics', 'predictions', 'figures']:
        safe_rmtree(seed_root / rel)

    graph_dir = Path(config['paths']['output_dir']) / 'graphs'
    graph_dir.mkdir(parents=True, exist_ok=True)
    graph_name = 'aem_graph.pt'
    src = BASE_GRAPH_DIR / graph_name
    dst = graph_dir / graph_name
    if not src.exists():
        raise FileNotFoundError(f'Missing parent graph: {src}')
    shutil.copy2(src, dst)

    checkpoint_path = train_experiment(config, EXPERIMENT_NAME)
    result = evaluate_experiment(config, EXPERIMENT_NAME)

    row = {
        'seed': seed,
        'checkpoint_path': str(checkpoint_path),
    }
    for split in SPLITS_TO_SUMMARIZE:
        for metric in METRICS_TO_SUMMARIZE:
            row[f'{split}_{metric}'] = metric_from_result(result, split, metric)
    run_rows.append(row)

run_summary = pd.DataFrame(run_rows)
run_summary.to_csv(ENSEMBLE_PARENT / 'ensemble_member_metrics.csv', index=False)
run_summary


## Aggregate Spatiotemporal Ensemble Predictions

We merge per-seed predictions by `sample_id`, verify they match the current spatiotemporal sample table, compute the ensemble mean and ensemble standard deviation, and then use the ensemble spread as a pointwise uncertainty scale.


In [ ]:
agg = None
pred_cols = []

for seed in SEEDS:
    seed_root = ENSEMBLE_PARENT / f'seed{seed}'
    pred_path = seed_root / 'predictions' / f'{EXPERIMENT_NAME}_predictions_all_splits.csv'
    frame = pd.read_csv(pred_path)
    pred_col = f'y_pred_seed_{seed}'
    pred_cols.append(pred_col)

    keep_cols = [
        'sample_id', 'grid_id', 'split', 'h_steps',
        'start_time_label', 'target_time_label',
        'y_true_delta_h_m', 'y_pred_delta_h_m'
    ]
    available_keep_cols = [col for col in keep_cols if col in frame.columns]
    sub = frame[available_keep_cols].rename(columns={'y_pred_delta_h_m': pred_col})
    if agg is None:
        agg = sub
    else:
        agg = agg.merge(sub[['sample_id', pred_col]], on='sample_id', how='inner')

agg['ensemble_mean_m'] = agg[pred_cols].mean(axis=1)
agg['ensemble_std_m'] = agg[pred_cols].std(axis=1, ddof=1).fillna(0.0)
agg['ensemble_residual_m'] = agg['y_true_delta_h_m'] - agg['ensemble_mean_m']

agg.head()


## Fit 75% Normalized Conformal Interval


In [ ]:
calibration_75 = fit_normalized_conformal(
    agg,
    fit_split='validation',
    pred_col='ensemble_mean_m',
    true_col='y_true_delta_h_m',
    scale_col='ensemble_std_m',
    levels=INTERVAL_LEVELS,
    eps=1.0e-6,
)

ensemble_with_pi = apply_normalized_conformal(
    agg,
    calibration_75,
    pred_col='ensemble_mean_m',
    scale_col='ensemble_std_m',
)

interval_summary_75 = summarize_prediction_intervals(
    ensemble_with_pi,
    true_col='y_true_delta_h_m',
    levels=INTERVAL_LEVELS,
)

interval_summary_75


## Save Outputs


In [ ]:
summary_dir = ENSEMBLE_PARENT / 'summary'
summary_dir.mkdir(parents=True, exist_ok=True)

calibration_75.to_csv(summary_dir / 'aem_gnn_normalized_conformal_calibration_75.csv', index=False)
ensemble_with_pi.to_csv(summary_dir / 'aem_gnn_ensemble_predictions_with_interval_75.csv', index=False)
interval_summary_75.to_csv(summary_dir / 'aem_gnn_interval_summary_75.csv', index=False)

print('Saved:', summary_dir / 'aem_gnn_interval_summary_75.csv')


## Inspect Pointwise Radii

In [ ]:
ensemble_with_pi[['sample_id', 'split', 'ensemble_mean_m', 'ensemble_std_m', 'pi_75_radius_m', 'pi_75_lower_m', 'pi_75_upper_m']].query("split == 'test'").head(20).reset_index(drop=True)
